In [3]:
# 代码作用：读取MES、LIMS数据汇总为生产实绩 v1
# 2026/03/16
# 数据治理工程师：yushumeng
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType,DateType
from pyspark.sql.functions import current_timestamp,split,lit,col
from pyspark.sql.functions import col, lit, sum, avg, count, when, round,concat
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import to_date
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()

# 读取MES班次交接数据
mes_bcjj_sql = """
WITH ranked_data AS (
    SELECT 
        T1.season,
        T1.factory,
        T1.hand_date,
        T1.team,
        T1.task_code,
        T2.project,
        CASE WHEN T2.value REGEXP '^[0-9.]+$' THEN CAST(T2.value AS DECIMAL(10,2)) ELSE NULL END AS value,
        ROW_NUMBER() OVER (
            PARTITION BY T1.season, T1.factory, T1.hand_date, T1.team, T2.project, T1.task_code
            ORDER BY T1.hand_time ASC
        ) AS rn
    FROM dwd.dwd_mes_pro_shift_change_record_f_1h T1 
    INNER JOIN dwd.dwd_mes_pro_shift_change_detail_f_1h T2 
        ON T1.hand_code = T2.record_code
    WHERE 
        T2.project = "当班放糖量(m³)：" 
        AND T1.task_code IN ('A0304','A0305')
        AND T1.flag_deleted = 0
),
unique_data AS (
    SELECT 
        season,
        factory,
        hand_date,
        team,
        task_code,
        value
    FROM ranked_data
    WHERE rn = 1
)
SELECT 
    season,
    factory,
    hand_date,
    team,
    SUM(CASE WHEN task_code = 'A0304' THEN value ELSE 0 END) AS b_syrup_vol_m3,
    SUM(CASE WHEN task_code = 'A0305' THEN value ELSE 0 END) AS c_syrup_vol_m3
FROM unique_data
GROUP BY season, factory, hand_date, team
ORDER BY hand_date DESC;
"""
mes_bcjj_df = spark.sql(mes_bcjj_sql)


# 读取MES生成记录数据
mes_scjl_sql = """
SELECT 
    T1.season,
    T1.factory,
    CONCAT('20', SUBSTRING(T1.record_code, 2, 2), '-', SUBSTRING(T1.record_code, 4, 2), '-', SUBSTRING(T1.record_code, 6, 2)) AS hand_date,
    T3.ord_class AS team,
    SUM(80) AS a_syrup_cube_volume,
    SUM(COALESCE(T2.evaporation_heating_syrup_quantity, 0)) AS evaporation_heating_syrup_quantity
FROM dwd.dwd_mes_pro_product_record_f_1h T1
INNER JOIN (
    SELECT DISTINCT
        sam_sample_original_no,
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND (ord_sample_name LIKE '%甲膏%')
        AND test_name1 IN ('锤度', '视纯度', '色值')
) T3 
    ON T1.season = T3.crushing_season 
    AND SUBSTRING(T1.record_code, POSITION('-' IN T1.record_code) + 1) = T3.sam_sample_original_no
LEFT JOIN (
    SELECT 
        produce_code,
        SUM(CASE WHEN material_name = '蒸发加热糖浆' THEN quantity ELSE 0 END) AS evaporation_heating_syrup_quantity
    FROM dwd.dwd_mes_prd_enter_data_f_1h
    GROUP BY produce_code
) T2 ON T1.record_code = T2.produce_code
WHERE T1.task_code = 'A0301' 
  AND T1.check_material_type = '甲膏'
GROUP BY T1.season, T1.factory, hand_date, T3.ord_class
ORDER BY hand_date DESC;
"""
mes_scjl_df = spark.sql(mes_scjl_sql)

mes_df = mes_bcjj_df.join(
    mes_scjl_df,
    on=["season","factory","hand_date","team"],
    how="left"
)


# 读取LIMS样品榨蔗桔水分析数据
lims_sql_js = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        test_name1,
        ord_up_ver,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_sample_name LIKE '%榨蔗桔水%'
        AND test_name1 IN ('重力纯度', '还原糖分')
),
syrup_data AS (
    SELECT 
        company AS syrup_company,
        crushing_season AS syrup_crushing_season,
        record_date AS syrup_record_date,
        work_shift AS syrup_work_shift,
        MIN(ord_up_ver) AS min_ord_up_ver,
        MAX(CASE WHEN test_name1 = '重力纯度' THEN test_value END) AS molasses_gravity_purity,
        MAX(CASE WHEN test_name1 = '还原糖分' THEN test_value END) AS molasses_reducing_sugar
    FROM raw_data
    GROUP BY company, crushing_season, record_date, work_shift
),
ranked_syrup_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY syrup_company, syrup_crushing_season, syrup_record_date, syrup_work_shift
            ORDER BY min_ord_up_ver ASC
        ) AS rn
    FROM syrup_data
)
SELECT 
    syrup_company,
    syrup_crushing_season,
    syrup_record_date,
    syrup_work_shift,
    molasses_gravity_purity,
    molasses_reducing_sugar
FROM ranked_syrup_data
WHERE rn = 1;
"""
lims_df_js = spark.sql(lims_sql_js)


# 读取LIMS班报榨蔗桔水
lims_js_sql ="""
SELECT 
    season,
    gongsidm,
    rq,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        ELSE banbiedm
    END AS banbiedm,
    chanliang,
    ap
FROM dwd.dwd_lims_sugar_cane_juice_batch_report_f_1h 
WHERE 
    gongsidm = 'FNR' 
    AND name = '榨蔗桔水'
    AND chanliang IS NOT NULL
    AND ap IS NOT NULL;
"""
lims_js_df = spark.sql(lims_js_sql)
lims_js_df = lims_js_df.withColumnRenamed("season", "js_season")


# 读取lims班报实绩数据
lims_bbsj_sql = """
SELECT 
season,
tb_gongsidm,
tb_rq,
CASE 
WHEN tb_banbiedm = '01' THEN '甲班'
WHEN tb_banbiedm = '02' THEN '乙班'
WHEN tb_banbiedm = '03' THEN '丙班'
ELSE '未知班别' 
END AS banbie_name,
sj_zhazheliang,
sj_zz_shengyuyuzhebi
FROM dwd.dwd_lims_batch_report_actual_data_f_1h WHERE tb_gongsidm = 'FNR';
"""
lims_bbsj_df = spark.sql(lims_bbsj_sql)
lims_bbsj_df = lims_bbsj_df.withColumnRenamed("season", "bbsj_season")

Spark 启动中...


In [4]:
# MES数据与班报实际
# 统一日期字段类型
lims_bbsj_df = lims_bbsj_df.withColumn("tb_rq", to_date(col("tb_rq")))
mes_df = mes_df.withColumn("hand_date", to_date(col("hand_date")))

# 执行左连接
mes_lims_df_v1 = lims_bbsj_df.join(
    mes_df,
    on=[
        lims_bbsj_df["bbsj_season"] == mes_df["season"],
        lims_bbsj_df["tb_gongsidm"] == mes_df["factory"],
        lims_bbsj_df["tb_rq"] == mes_df["hand_date"],
        lims_bbsj_df["banbie_name"] == mes_df["team"]
    ],
    how="left"
)
mes_lims_df_v1 = mes_lims_df_v1.select([
    "bbsj_season", 
    "tb_gongsidm", 
    "tb_rq", 
    "banbie_name", 
    "sj_zhazheliang", 
    "evaporation_heating_syrup_quantity",
    "sj_zz_shengyuyuzhebi", 
    "a_syrup_cube_volume",
    "b_syrup_vol_m3", 
    "c_syrup_vol_m3", 
])
# 获取LIMS样品榨蔗桔水分析数据
# 统一日期字段类型
mes_lims_df_v1 = mes_lims_df_v1.withColumn("tb_rq", to_date(col("tb_rq")))
lims_df_js = lims_df_js.withColumn("syrup_record_date", to_date(col("syrup_record_date")))

# 执行左连接
mes_lims_df_v2 = mes_lims_df_v1.join(
    lims_df_js,
    on=[
        mes_lims_df_v1["bbsj_season"] == lims_df_js["syrup_crushing_season"],
        mes_lims_df_v1["tb_gongsidm"] == lims_df_js["syrup_company"],
        mes_lims_df_v1["tb_rq"] == lims_df_js["syrup_record_date"],
        mes_lims_df_v1["banbie_name"] == lims_df_js["syrup_work_shift"]
    ],
    how="left"
)
mes_lims_df_v2 = mes_lims_df_v2.select([
    "bbsj_season", 
    "tb_gongsidm", 
    "tb_rq", 
    "banbie_name", 
    "sj_zhazheliang", 
    "evaporation_heating_syrup_quantity",
    "sj_zz_shengyuyuzhebi", 
    "a_syrup_cube_volume",
    "b_syrup_vol_m3", 
    "c_syrup_vol_m3",
    "molasses_gravity_purity"
])
# 获取LIMS样品榨蔗桔水分析数据
# 统一日期字段类型
mes_lims_df_v2 = mes_lims_df_v2.withColumn("tb_rq", to_date(col("tb_rq")))
lims_js_df = lims_js_df.withColumn("rq", to_date(col("rq")))

# 执行左连接
mes_lims_df_v3 = mes_lims_df_v2.join(
    lims_js_df,
    on=[
        mes_lims_df_v2["bbsj_season"] == lims_js_df["js_season"],
        mes_lims_df_v2["tb_gongsidm"] == lims_js_df["gongsidm"],
        mes_lims_df_v2["tb_rq"] == lims_js_df["rq"],
        mes_lims_df_v2["banbie_name"] == lims_js_df["banbiedm"]
    ],
    how="left"
)
mes_lims_df_v3 = mes_lims_df_v3.select([
    "bbsj_season", 
    "tb_gongsidm", 
    "tb_rq", 
    "banbie_name", 
    "sj_zhazheliang", 
    "evaporation_heating_syrup_quantity",
    "sj_zz_shengyuyuzhebi", 
    "a_syrup_cube_volume",
    "b_syrup_vol_m3", 
    "c_syrup_vol_m3", 
    "molasses_gravity_purity",
    "chanliang"
])

In [5]:
# ===================== 写入Hive表 - 煮糖生产实绩 =====================
from pyspark.sql.types import StructType, StructField, StringType, DateType, DecimalType, TimestampType
from pyspark.sql.functions import col, lit
from datetime import datetime

# 目标表名
target_table = "app.app_mes_boiling_sugar_production_actual_f_1h"

# 定义Schema（与您的字段完全一致）
target_schema = StructType([
    StructField("bbsj_season", StringType(), True),
    StructField("tb_gongsidm", StringType(), True),
    StructField("tb_rq", DateType(), True),
    StructField("banbie_name", StringType(), True),
    StructField("sj_zhazheliang", DecimalType(20,4), True),
    StructField("evaporation_heating_syrup_quantity", DecimalType(20,4), True),
    StructField("sj_zz_shengyuyuzhebi", DecimalType(20,4), True),
    StructField("a_syrup_cube_volume", DecimalType(20,4), True),
    StructField("b_syrup_vol_m3", DecimalType(20,4), True),
    StructField("c_syrup_vol_m3", DecimalType(20,4), True),
    StructField("molasses_gravity_purity", DecimalType(20,4), True),
    StructField("chanliang", DecimalType(20,4), True),
    StructField("source_flg", StringType(), True),
    StructField("dw_update_time", TimestampType(), True)
])

# 添加元数据字段
mes_lims_df_v3_with_meta = mes_lims_df_v3 \
    .withColumn("source_flg", lit("LIMS+MES")) \
    .withColumn("dw_update_time", lit(datetime.now()))

# 类型转换并写入
select_exprs = []
for field in target_schema.fields:
    if field.name in mes_lims_df_v3_with_meta.columns:
        select_exprs.append(col(field.name).cast(field.dataType).alias(field.name))
    else:
        select_exprs.append(lit(None).cast(field.dataType).alias(field.name))

df_to_write = mes_lims_df_v3_with_meta.select(*select_exprs)

# 写入Hive表
df_to_write.write \
    .mode("overwrite") \
    .format("hive") \
    .option("encoding", "UTF-8") \
    .saveAsTable(target_table)

# 添加表级注释
spark.sql(f"""
    ALTER TABLE {target_table} 
    SET TBLPROPERTIES (
        'comment' = '煮糖生产实绩',
        'create_time' = '{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
    )
""")

# 字段注释（以您提供的为准）
field_comments = {
    "bbsj_season": "榨季",
    "tb_gongsidm": "公司代码",
    "tb_rq": "日期",
    "banbie_name": "班别",
    "sj_zhazheliang": "榨蔗量T",
    "evaporation_heating_syrup_quantity": "粗糖浆量m³",
    "sj_zz_shengyuyuzhebi": "蔗渣剩余率",
    "a_syrup_cube_volume": "A糖膏立方数m³",
    "b_syrup_vol_m3": "B糖膏立方数m³",
    "c_syrup_vol_m3": "C糖膏立方数m³",
    "molasses_gravity_purity": "桔水重力纯度",
    "chanliang": "桔水产量T",
    "source_flg": "数据来源标识（LIMS+MES）",
    "dw_update_time": "数仓更新时间"
}

# 工具函数：获取Hive类型字符串
def get_hive_type_str(field):
    dt = field.dataType
    if isinstance(dt, StringType):
        return "STRING"
    elif isinstance(dt, DateType):
        return "DATE"
    elif isinstance(dt, TimestampType):
        return "TIMESTAMP"
    elif isinstance(dt, DecimalType):
        return f"DECIMAL({dt.precision},{dt.scale})"
    else:
        return "STRING"

# 添加字段注释
field_type_map = {f.name: get_hive_type_str(f) for f in target_schema.fields}
for field, comment in field_comments.items():
    spark.sql(f"""
        ALTER TABLE {target_table} 
        CHANGE COLUMN {field} {field} {field_type_map[field]} COMMENT '{comment}'
    """)

print(f"✅ 数据已写入：{target_table}")
print(f"🕒 {datetime.now()}")

✅ 数据已写入：app.app_mes_boiling_sugar_production_actual_f_1h
🕒 2026-03-18 11:02:52.592270
